#  00 SOURCE SETUP
##  `orders.csv` → Unity Catalog Volume

This is the first notebook in the simple demo pipeline.

```text
project file
orders.csv
    ↓
Managed Unity Catalog Volume
    ↓
Ready for Bronze ingestion
```

Run order:

```text
00_SOURCE_SETUP
01_BRONZE
02_SILVER
03_GOLD
```


In [0]:
# ============================================================
# CELL 1 — FIND THE OFFICIAL TRAINING PROJECT
# ============================================================
# WHAT:
#   Locate the existing adbnov2026-Dev project and orders.csv.
#
# WHY:
#   We use only the training dataset.
#
# EXPECTED:
#   orders.csv exists.

from pathlib import Path
import os
import shutil

def find_project_root():
    cwd = Path(os.getcwd()).resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "RETAIL_LAB.json").exists():
            return candidate

    for root in [
        Path("/Workspace/Users"),
        Path("/Workspace/Shared"),
        Path("/Workspace"),
    ]:
        if not root.exists():
            continue

        for current_root, dirs, files in os.walk(root):
            dirs[:] = [
                d for d in dirs
                if not d.startswith(".")
                and d not in {"Trash", ".Trash"}
            ]
            if "RETAIL_LAB.json" in files:
                return Path(current_root)

    raise FileNotFoundError(
        "Could not locate RETAIL_LAB.json. "
        "Import these demo notebooks inside adbnov2026-Dev."
    )

PROJECT_ROOT = find_project_root()

orders_candidates = [
    PROJECT_ROOT / "data" / "official_sources" / "orders.csv",
    PROJECT_ROOT / "data" / "orders.csv",
]

ORDERS_SOURCE = next(
    (p for p in orders_candidates if p.exists()),
    None,
)

if ORDERS_SOURCE is None:
    raise FileNotFoundError("Official orders.csv not found.")

print("Project root :", PROJECT_ROOT)
print("Orders file  :", ORDERS_SOURCE)


In [0]:
# ============================================================
# CELL 2 — RESOLVE UNITY CATALOG NAMESPACES
# ============================================================
# WHAT:
#   Use the current Unity Catalog and create/reuse Bronze, Silver and Gold schemas.
#
# WHY:
#   The demo should visually show the Medallion layers.
#
# EXPECTED:
#   BRONZE_SCHEMA, SILVER_SCHEMA and GOLD_SCHEMA are resolved.

ctx = spark.sql("""
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema
""").first()

CATALOG = ctx["catalog"]
CURRENT_SCHEMA = ctx["schema"]

if CATALOG == "hive_metastore":
    raise RuntimeError(
        "Switch to a Unity Catalog catalog before running this demo."
    )

def qi(value: str) -> str:
    return f"`{value.replace('`', '``')}`"

def schema_fqn(schema: str) -> str:
    return f"{qi(CATALOG)}.{qi(schema)}"

preferred = {
    "bronze": "retail_bronze",
    "silver": "retail_silver",
    "gold": "retail_gold",
}

resolved = {}

for layer, schema_name in preferred.items():
    try:
        spark.sql(
            f"CREATE SCHEMA IF NOT EXISTS {schema_fqn(schema_name)}"
        )
        resolved[layer] = schema_name
    except Exception:
        resolved[layer] = CURRENT_SCHEMA

BRONZE_SCHEMA = resolved["bronze"]
SILVER_SCHEMA = resolved["silver"]
GOLD_SCHEMA = resolved["gold"]

print("Catalog:", CATALOG)
print("Bronze :", BRONZE_SCHEMA)
print("Silver :", SILVER_SCHEMA)
print("Gold   :", GOLD_SCHEMA)


In [0]:
# ============================================================
# CELL 3 — CREATE A MANAGED SOURCE VOLUME
# ============================================================
# WHAT:
#   Create a governed Volume for the demo source file.
#
# WHY:
#   Students should see a realistic path:
#       project file -> governed source Volume -> Bronze table
#
# EXPECTED:
#   A managed Volume named demo_source_files exists.

VOLUME_NAME = "demo_source_files"

VOLUME_FQN = (
    f"{qi(CATALOG)}."
    f"{qi(BRONZE_SCHEMA)}."
    f"{qi(VOLUME_NAME)}"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {VOLUME_FQN}"
)

VOLUME_ROOT = (
    f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
)

SOURCE_FOLDER = f"{VOLUME_ROOT}/orders"
SOURCE_FILE = f"{SOURCE_FOLDER}/orders.csv"

Path(SOURCE_FOLDER).mkdir(
    parents=True,
    exist_ok=True,
)

print("Volume path:", VOLUME_ROOT)


In [0]:
# ============================================================
# CELL 4 — COPY  orders.csv INTO THE VOLUME
# ============================================================
# WHAT:
#   Copy the orders.csv into the governed Volume.
#
# WHY:
#   This simulates a source file landing in a data-engineering landing zone.
#
# EXPECTED:
#   orders.csv appears under the Volume path.

shutil.copy2(
    ORDERS_SOURCE,
    Path(SOURCE_FILE),
)

display(
    dbutils.fs.ls(
        SOURCE_FOLDER
    )
)


In [0]:
# ============================================================
# CELL 5 — FINAL SOURCE CHECK
# ============================================================
# WHAT:
#   Read the CSV once only to verify the source row count.
#
# WHY:
#   We need a known starting point before Bronze ingestion.
#
# EXPECTED:
#   126 source rows.

source_count = (
    spark.read
    .option("header", True)
    .csv(SOURCE_FILE)
    .count()
)

print("Source rows:", source_count)

assert source_count == 126

print("\nSOURCE SETUP: PASSED")
print("Next notebook: 01_BRONZE")
